# 06 — Recovery-bias validation

**Goal:** demonstrate `kinextract.assess_recovery_bias` and
`kinextract.correct_recovered_losvd` -- the recommended way to
characterize and (optionally) correct for LOSVD recovery bias on a
*specific* target, rather than relying on the generic multi-instrument
numbers in `FitConfig`'s "Known limitations" docstring section.

This matters most near the instrumental resolution limit (target sigma
comparable to or below the instrument's own LSF sigma), where the
default MAP+bootstrap pipeline (see notebook 01) has a real, but
condition-dependent, bias -- not one-signed or uniform enough across
instruments/LOSVD shapes to correct with a single rule of thumb.

---
## Method overview

1. Build and fit a synthetic galaxy spectrum near its instrument's own
   resolution limit (same self-contained mock-generation approach as
   notebook 01, but with a smaller sigma_true so the bias is visible).
2. Run `assess_recovery_bias`: for a small grid of known (V, sigma)
   truths, it generates matched mock spectra (same instrument
   resolution, template mixture, continuum, and noise level as the fit
   above), refits each with the same MAP+bootstrap pipeline, and reports
   the empirical `recovered - true` bias.
3. Apply `correct_recovered_losvd` to bias-correct the original fit's
   recovered V/sigma, with an inflated uncertainty reflecting the bias
   table's own seed-to-seed scatter.

In [1]:
from __future__ import annotations
import tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, shift as ndimage_shift

from kinextract import (
    FitConfig, run_spectral_fit, assess_recovery_bias, correct_recovered_losvd, set_verbose,
)
from kinextract.losvd import fit_losvd_gauss_hermite

set_verbose(False)  # silence kinextract's internal progress logging

print('kinextract imported OK')

kinextract imported OK


## 1. Build a mock spectrum near the instrument's resolution limit

Same MUSE-like CaII triplet setup as notebook 01, but with an
instrumental LSF now modeled explicitly (`LSF_SIGMA_KMS`) and a
`TRUE_SIGMA` deliberately chosen close to it -- the regime where
recovery bias is largest and where `assess_recovery_bias` earns its
extra runtime.

In [2]:
WAVEMIN    = 4750.0     # Å — full spectrum start
STEP       = 1.25       # Å — pixel size
N_PIX      = 3681       # pixels in full spectrum
WAVEFITMIN = 8400.0     # Å — fit range start
WAVEFITMAX = 8800.0     # Å — fit range end
LAM_CENTER = 8580.0     # Å — representative wavelength
CEE        = 299792.458 # km/s

LSF_SIGMA_KMS = 42.0     # MUSE WFM, R~3000 near CaII

TRUE_V     =  30.0  # km/s
TRUE_SIGMA =  45.0  # km/s -- close to LSF_SIGMA_KMS: near the resolution limit

wavelength = WAVEMIN + np.arange(N_PIX) * STEP

CA_CENTERS = [8498.02, 8542.09, 8662.14]
CA_DEPTHS  = [0.55, 0.70, 0.65]
template = np.ones(N_PIX)
for cen, depth in zip(CA_CENTERS, CA_DEPTHS):
    template -= depth * np.exp(-0.5 * ((wavelength - cen) / 5.0) ** 2)

sigma_pix     = TRUE_SIGMA * LAM_CENTER / (CEE * STEP)
shift_pix     = TRUE_V     * LAM_CENTER / (CEE * STEP)
lsf_sigma_pix = LSF_SIGMA_KMS * LAM_CENTER / (CEE * STEP)

broadened = gaussian_filter(template, sigma=sigma_pix)
galaxy    = ndimage_shift(broadened, shift=+shift_pix)
galaxy    = gaussian_filter(galaxy, sigma=lsf_sigma_pix)  # instrumental LSF

RNG    = np.random.default_rng(42)
NOISE  = 0.02
galaxy = galaxy + RNG.normal(0.0, NOISE, N_PIX)
errors = np.full(N_PIX, NOISE)

print(f'TRUE_SIGMA / LSF_SIGMA_KMS = {TRUE_SIGMA / LSF_SIGMA_KMS:.2f}x  (near the resolution limit)')

TRUE_SIGMA / LSF_SIGMA_KMS = 1.07x  (near the resolution limit)


## 2. Fit it with the default MAP+bootstrap pipeline

`data_fwhm_A`/`template_fwhm_A` tell kinextract's own LSF-matching
machinery that the galaxy spectrum carries the instrumental LSF above
but the template is sharp -- the realistic case for real data, where the
template library and the science data rarely share a native resolution.

In [3]:
tmpdir    = Path(tempfile.mkdtemp(prefix='kinextract_nb06_'))
spec_path = tmpdir / 'mock_galaxy.spec'
tmpl_path = tmpdir / 'mock_template.dat'

np.savetxt(spec_path, np.column_stack([np.arange(1, N_PIX+1), galaxy, errors]),
           fmt='%6d  %14.8f  %14.8f')
np.savetxt(tmpl_path, np.column_stack([wavelength, template, np.full(N_PIX, 0.001)]),
           fmt='%10.4f  %14.8f  %12.8f')
(tmpdir / 'Tlist').write_text('mock_template.dat\n')

lsf_fwhm_A = LSF_SIGMA_KMS * (2.0 * np.sqrt(2.0 * np.log(2.0))) * LAM_CENTER / CEE

cfg = FitConfig(
    template_list_file  = str(tmpdir / 'Tlist'),
    template_dir        = str(tmpdir),
    # outdir=str(tmpdir), write_outputs=True,  # uncomment to save .fit/.temp/.ascii/.rms output files
    wavemin_full        = WAVEMIN,
    step                = STEP,
    wavefitmin          = WAVEFITMIN,
    wavefitmax          = WAVEFITMAX,
    zgal                = 0.0,
    fit_als_continuum   = False,
    data_fwhm_A         = lsf_fwhm_A,
    template_fwhm_A     = 0.05,
    data_fwhm_frame      = 'rest',
    xlam_auto           = True,
    xlam_criterion       = 'chi2',
    xlam_chi2_tolerance  = 0.02,
    xlam_auto_grid      = (10., 100., 1000., 10000., 100000.),
    xlam_max_peaks      = 1,
    sigl                = TRUE_SIGMA,
    losvd_vmin          = -4.5 * TRUE_SIGMA,
    losvd_vmax          = +4.5 * TRUE_SIGMA,
    use_spectrum_errors = True,
    clean               = False,
    map_maxiter         = 5000,
    print_every         = 999999,
)

fit = run_spectral_fit(cfg, gal_file=str(spec_path))
st  = fit['state']
gh  = fit_losvd_gauss_hermite(st.xl, fit['outputs']['b'], fit_h3h4=True)

print(f"success : {fit['result'].success}")
print(f"recovered V     = {gh['vherm']:+.2f} km/s  (true {TRUE_V:+.1f})")
print(f"recovered sigma = {gh['sherm']:.2f} km/s  (true {TRUE_SIGMA:.1f})")

success : True
recovered V     = +24.27 km/s  (true +30.0)
recovered sigma = 53.57 km/s  (true 45.0)


## 3. Assess recovery bias on a matched mock grid

`assess_recovery_bias` reuses `fit`'s own instrument grid, template
mixture, continuum, and per-pixel noise level (via
`kinextract.mocks.build_matched_mock`) to generate mocks with *known*
ground truth, refits each, and reports the empirical bias -- this is
the actual pipeline behavior for *this* target's specific configuration,
not a generic multi-instrument average.

A small grid and few seeds keep this notebook fast; for a real
publication-quality bias table, use more seeds (`n_seeds=20-50`) and a
finer grid around the target's expected V/sigma.

In [4]:
bias_table = assess_recovery_bias(
    fit, cfg,
    v_true_grid=[0.0, TRUE_V],
    sigma_true_grid=[TRUE_SIGMA, 2 * TRUE_SIGMA],
    n_seeds=5, seed0=1000,
)

print(f"{'(v_true, sigma_true)':22s} {'bias_V':>10s} {'bias_sigma':>12s} {'n_seeds':>8s}")
for (v_true, sigma_true), entry in sorted(bias_table.items()):
    print(f"({v_true:6.1f}, {sigma_true:6.1f})   "
          f"{entry['bias_v']:+7.2f}   {entry['bias_sigma']:+9.2f}   {entry['n_seeds']:8d}")

(v_true, sigma_true)       bias_V   bias_sigma  n_seeds
(   0.0,   45.0)     +0.02       +5.73          5
(   0.0,   90.0)     +0.55       +3.91          5
(  30.0,   45.0)     -3.92       +5.86          5
(  30.0,   90.0)     +3.26       +0.94          5


## 4. Apply the correction

`correct_recovered_losvd` interpolates the bias table at the real fit's
own recovered (V, sigma) and returns a bias-corrected point estimate
plus an inflated uncertainty from the bias table's own seed-to-seed
scatter -- an empirically-calibrated alternative to the retired analytic
`kinextract.errors.bias_corrected_losvd`, which is unreliable in exactly
this regime (see `FitConfig`'s "Known limitations" section).

In [5]:
corrected = correct_recovered_losvd(gh['vherm'], gh['sherm'], bias_table)

print('Raw recovered:')
print(f"  V     = {gh['vherm']:+.2f} km/s   (true {TRUE_V:+.1f})")
print(f"  sigma = {gh['sherm']:.2f} km/s   (true {TRUE_SIGMA:.1f})")
print('Bias-corrected:')
print(f"  V     = {corrected['v_corrected']:+.2f} +/- {corrected['v_uncertainty_inflation']:.2f} km/s")
print(f"  sigma = {corrected['sigma_corrected']:.2f} +/- {corrected['sigma_uncertainty_inflation']:.2f} km/s")

Raw recovered:
  V     = +24.27 km/s   (true +30.0)
  sigma = 53.57 km/s   (true 45.0)
Bias-corrected:
  V     = +26.07 +/- 1.67 km/s
  sigma = 48.67 +/- 5.03 km/s


## Summary

- Use this workflow when a target's expected sigma is within about 2x
  the instrument's LSF sigma -- well away from the resolution limit, the
  bias shrinks toward zero and the extra mock-fitting runtime isn't
  worth it.
- The bias-corrected uncertainty (`v_uncertainty_inflation`/
  `sigma_uncertainty_inflation`) reflects only the bias table's own
  Monte Carlo noise -- combine it in quadrature with
  `LOSVDErrorEstimator`'s bootstrap/Laplace uncertainty for the target's
  actual data (see notebook 03's "Error estimation" section), rather than
  using it as a standalone error bar.
- `correct_recovered_losvd` is always opt-in -- `run_spectral_fit()`
  never applies this correction automatically.